In [1]:
!pip install torch pandas scikit-learn

In [2]:
from google.colab import files
uploaded = files.upload()
# upload ml-1m.zip when prompted

Saving ml-1m.zip to ml-1m.zip


In [3]:
import zipfile

with zipfile.ZipFile("ml-1m.zip", "r") as z:
    z.extractall(".")

print("Extracted files:")
import os
for f in os.listdir("ml-1m"):
    print(f)

Extracted files:
ratings.dat
users.dat
README
movies.dat


In [4]:
import pandas as pd

df = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    header=None,
    names=["user_id", "item_id", "rating", "timestamp"],
    engine="python"
)

print(f"Total interactions: {len(df):,}")
print(df.head())

Total interactions: 1,000,209
   user_id  item_id  rating  timestamp
0        1     1193       5  978300760
1        1      661       3  978302109
2        1      914       3  978301968
3        1     3408       4  978300275
4        1     2355       5  978824291


In [5]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train size : {len(train_df):,}")
print(f"Val size   : {len(val_df):,}")
print(f"Test size  : {len(test_df):,}")

Train size : 800,167
Val size   : 100,021
Test size  : 100,021


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Encode user/item ids to sequential indices ────────────────────────────────
user2idx = {u: i for i, u in enumerate(df["user_id"].unique())}
item2idx = {it: i for i, it in enumerate(df["item_id"].unique())}

n_users = len(user2idx)
n_items = len(item2idx)
print(f"Users: {n_users} | Items: {n_items}")

# ── Dataset ───────────────────────────────────────────────────────────────────
class RatingDataset(Dataset):
    def __init__(self, df):
        self.users  = torch.tensor([user2idx[u] for u in df["user_id"]], dtype=torch.long)
        self.items  = torch.tensor([item2idx[i] for i in df["item_id"]], dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

train_loader = DataLoader(RatingDataset(train_df), batch_size=1024, shuffle=True)
test_loader  = DataLoader(RatingDataset(test_df),  batch_size=1024, shuffle=False)

# ── Model ─────────────────────────────────────────────────────────────────────
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)

    def forward(self, users, items):
        dot  = (self.user_emb(users) * self.item_emb(items)).sum(dim=1)
        bias = self.user_bias(users).squeeze() + self.item_bias(items).squeeze()
        return dot + bias

# ── Train ─────────────────────────────────────────────────────────────────────
model     = MatrixFactorization(n_users, n_items, n_factors=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()

EPOCHS = 50
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        preds = model(users, items)
        loss  = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")



Using device: cpu
Users: 6040 | Items: 3706
Epoch 1/50 | Loss: 79.4904


KeyboardInterrupt: 

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
model.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for users, items, ratings in test_loader:
        users, items = users.to(device), items.to(device)
        preds = model(users, items).cpu().numpy()
        preds_all.extend(preds)
        labels_all.extend(ratings.numpy())

preds_all  = np.array(preds_all)
labels_all = np.array(labels_all)

svd_rmse = np.sqrt(((preds_all - labels_all) ** 2).mean())
svd_mae  = np.abs(preds_all - labels_all).mean()

print(f"\nSVD → RMSE: {svd_rmse:.4f} | MAE: {svd_mae:.4f}")

In [8]:
svd_results = {"model": "SVD", "RMSE": round(svd_rmse, 4), "MAE": round(svd_mae, 4)}
print(svd_results)

{'model': 'SVD', 'RMSE': np.float32(0.9609), 'MAE': np.float32(0.7493)}


In [9]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, layers=[128, 64, 32]):
        super().__init__()
        # GMF embeddings
        self.gmf_user = nn.Embedding(n_users, n_factors)
        self.gmf_item = nn.Embedding(n_items, n_factors)

        # MLP embeddings
        self.mlp_user = nn.Embedding(n_users, n_factors)
        self.mlp_item = nn.Embedding(n_items, n_factors)

        # MLP layers
        mlp_input_size = n_factors * 2
        mlp_layers = []
        for out_size in layers:
            mlp_layers.append(nn.Linear(mlp_input_size, out_size))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(0.2))
            mlp_input_size = out_size
        self.mlp = nn.Sequential(*mlp_layers)

        # Final prediction layer
        self.output = nn.Linear(n_factors + layers[-1], 1)

    def forward(self, users, items):
        # GMF branch
        gmf = self.gmf_user(users) * self.gmf_item(items)

        # MLP branch
        mlp_input = torch.cat([self.mlp_user(users), self.mlp_item(items)], dim=1)
        mlp_out = self.mlp(mlp_input)

        # Concat and predict
        out = torch.cat([gmf, mlp_out], dim=1)
        return self.output(out).squeeze()

# ── Train ─────────────────────────────────────────────────────────────────────
ncf_model = NCF(n_users, n_items, n_factors=64, layers=[128, 64, 32]).to(device)
optimizer  = torch.optim.Adam(ncf_model.parameters(), lr=0.001, weight_decay=1e-5)
criterion  = nn.MSELoss()

EPOCHS = 50
for epoch in range(EPOCHS):
    ncf_model.train()
    total_loss = 0
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        preds = ncf_model(users, items)
        loss  = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
ncf_model.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for users, items, ratings in test_loader:
        users, items = users.to(device), items.to(device)
        preds = ncf_model(users, items).cpu().numpy()
        preds_all.extend(preds)
        labels_all.extend(ratings.numpy())

preds_all  = np.array(preds_all)
labels_all = np.array(labels_all)

ncf_rmse = np.sqrt(((preds_all - labels_all) ** 2).mean())
ncf_mae  = np.abs(preds_all - labels_all).mean()

print(f"\nNCF → RMSE: {ncf_rmse:.4f} | MAE: {ncf_mae:.4f}")

Epoch 1/50 | Loss: 1.7708
Epoch 2/50 | Loss: 1.1778
Epoch 3/50 | Loss: 1.0726
Epoch 4/50 | Loss: 1.0018
Epoch 5/50 | Loss: 0.9546
Epoch 6/50 | Loss: 0.9205
Epoch 7/50 | Loss: 0.8883
Epoch 8/50 | Loss: 0.8635
Epoch 9/50 | Loss: 0.8333
Epoch 10/50 | Loss: 0.7859
Epoch 11/50 | Loss: 0.7532
Epoch 12/50 | Loss: 0.7214
Epoch 13/50 | Loss: 0.6996
Epoch 14/50 | Loss: 0.6771
Epoch 15/50 | Loss: 0.6556
Epoch 16/50 | Loss: 0.6399
Epoch 17/50 | Loss: 0.6280
Epoch 18/50 | Loss: 0.6168
Epoch 19/50 | Loss: 0.6071
Epoch 20/50 | Loss: 0.5976
Epoch 21/50 | Loss: 0.5891
Epoch 22/50 | Loss: 0.5817
Epoch 23/50 | Loss: 0.5740
Epoch 24/50 | Loss: 0.5666
Epoch 25/50 | Loss: 0.5596
Epoch 26/50 | Loss: 0.5538
Epoch 27/50 | Loss: 0.5463
Epoch 28/50 | Loss: 0.5395
Epoch 29/50 | Loss: 0.5331
Epoch 30/50 | Loss: 0.5272
Epoch 31/50 | Loss: 0.5232
Epoch 32/50 | Loss: 0.5178
Epoch 33/50 | Loss: 0.5130
Epoch 34/50 | Loss: 0.5090
Epoch 35/50 | Loss: 0.5051
Epoch 36/50 | Loss: 0.5004
Epoch 37/50 | Loss: 0.4962
Epoch 38/5

In [10]:
ncf_results = {"model": "NCF", "RMSE": round(ncf_rmse, 4), "MAE": round(ncf_mae, 4)}
print(ncf_results)

{'model': 'NCF', 'RMSE': np.float32(0.9127), 'MAE': np.float32(0.7063)}


In [11]:
import scipy.sparse as sp

class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers

        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def build_adj_matrix(self, df, device):
        users = df["user_id"].map(user2idx).values
        items = df["item_id"].map(item2idx).values

        # Build sparse adjacency matrix
        n = self.n_users + self.n_items
        row = np.concatenate([users, items + self.n_users])
        col = np.concatenate([items + self.n_users, users])
        data = np.ones(len(row))

        adj = sp.coo_matrix((data, (row, col)), shape=(n, n))

        # Normalize: D^-1/2 * A * D^-1/2
        deg  = np.array(adj.sum(axis=1)).flatten()
        d_inv = np.power(deg, -0.5)
        d_inv[np.isinf(d_inv)] = 0
        D = sp.diags(d_inv)
        adj = D @ adj @ D

        # Convert to torch sparse tensor
        adj = adj.tocoo()
        indices = torch.tensor(np.array([adj.row, adj.col]), dtype=torch.long)
        values  = torch.tensor(adj.data, dtype=torch.float32)
        return torch.sparse_coo_tensor(indices, values, (n, n)).to(device)

    def forward(self, adj):
        embs = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        all_embs = [embs]

        for _ in range(self.n_layers):
            embs = torch.sparse.mm(adj, embs)
            all_embs.append(embs)

        all_embs = torch.stack(all_embs, dim=1).mean(dim=1)
        return all_embs[:self.n_users], all_embs[self.n_users:]

    def predict(self, users, items, adj):
        user_embs, item_embs = self.forward(adj)
        return (user_embs[users] * item_embs[items]).sum(dim=1)

# ── Build adjacency matrix from train set ─────────────────────────────────────
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
adj = lgcn_model.build_adj_matrix(train_df, device)

# ── Train ─────────────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(lgcn_model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()

EPOCHS = 50
for epoch in range(EPOCHS):
    lgcn_model.train()
    total_loss = 0
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        preds = lgcn_model.predict(users, items, adj)
        loss  = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
lgcn_model.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for users, items, ratings in test_loader:
        users, items = users.to(device), items.to(device)
        preds = lgcn_model.predict(users, items, adj).cpu().numpy()
        preds_all.extend(preds)
        labels_all.extend(ratings.numpy())

preds_all  = np.array(preds_all)
labels_all = np.array(labels_all)

lgcn_rmse = np.sqrt(((preds_all - labels_all) ** 2).mean())
lgcn_mae  = np.abs(preds_all - labels_all).mean()

print(f"\nLightGCN → RMSE: {lgcn_rmse:.4f} | MAE: {lgcn_mae:.4f}")

/tmp/ipykernel_3978/638447186.py:30: RuntimeWarning: divide by zero encountered in power
  d_inv = np.power(deg, -0.5)


Epoch 1/50 | Loss: 3.8311
Epoch 2/50 | Loss: 1.8384
Epoch 3/50 | Loss: 1.4696
Epoch 4/50 | Loss: 1.2768
Epoch 5/50 | Loss: 1.1671
Epoch 6/50 | Loss: 1.1022
Epoch 7/50 | Loss: 1.0628
Epoch 8/50 | Loss: 1.0385
Epoch 9/50 | Loss: 1.0232
Epoch 10/50 | Loss: 1.0134
Epoch 11/50 | Loss: 1.0074
Epoch 12/50 | Loss: 1.0032
Epoch 13/50 | Loss: 1.0005
Epoch 14/50 | Loss: 0.9990
Epoch 15/50 | Loss: 0.9976
Epoch 16/50 | Loss: 0.9967
Epoch 17/50 | Loss: 0.9959
Epoch 18/50 | Loss: 0.9959
Epoch 19/50 | Loss: 0.9948
Epoch 20/50 | Loss: 0.9940
Epoch 21/50 | Loss: 0.9932
Epoch 22/50 | Loss: 0.9921
Epoch 23/50 | Loss: 0.9906
Epoch 24/50 | Loss: 0.9888
Epoch 25/50 | Loss: 0.9859
Epoch 26/50 | Loss: 0.9833
Epoch 27/50 | Loss: 0.9799
Epoch 28/50 | Loss: 0.9768
Epoch 29/50 | Loss: 0.9741
Epoch 30/50 | Loss: 0.9712
Epoch 31/50 | Loss: 0.9694
Epoch 32/50 | Loss: 0.9675
Epoch 33/50 | Loss: 0.9659
Epoch 34/50 | Loss: 0.9640
Epoch 35/50 | Loss: 0.9626
Epoch 36/50 | Loss: 0.9608
Epoch 37/50 | Loss: 0.9590
Epoch 38/5

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Keep only ratings >= 4 as positive interactions
df_implicit = df[df["rating"] >= 4].copy()
df_implicit["label"] = 1

print(f"Original interactions : {len(df):,}")
print(f"Positive interactions : {len(df_implicit):,}")

# Split
train_df, temp_df = train_test_split(df_implicit, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"\nTrain : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")

# Re-encode user/item ids
user2idx = {u: i for i, u in enumerate(df_implicit["user_id"].unique())}
item2idx = {it: i for i, it in enumerate(df_implicit["item_id"].unique())}

n_users = len(user2idx)
n_items = len(item2idx)

print(f"\nUsers : {n_users}")
print(f"Items : {n_items}")

Original interactions : 1,000,209
Positive interactions : 575,281

Train : 460,224
Val   : 57,528
Test  : 57,529

Users : 6038
Items : 3533


In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Dataset ───────────────────────────────────────────────────────────────────
class ImplicitDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor([user2idx[u] for u in df["user_id"]], dtype=torch.long)
        self.items = torch.tensor([item2idx[i] for i in df["item_id"]], dtype=torch.long)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx]

train_loader = DataLoader(ImplicitDataset(train_df), batch_size=1024, shuffle=True)

# ── Negative sampling for BPR loss ───────────────────────────────────────────
train_user_items = train_df.groupby("user_id")["item_id"].apply(set).to_dict()
all_items = set(item2idx.keys())

def sample_negatives(users_raw, n_neg=1):
    neg_items = []
    for u in users_raw:
        pos = train_user_items.get(u, set())
        negs = list(all_items - pos)
        neg_items.append(item2idx[np.random.choice(negs)])
    return torch.tensor(neg_items, dtype=torch.long)

# ── Ranking Evaluation ────────────────────────────────────────────────────────
def evaluate(model, test_df, K=10, model_type="mf"):
    model.eval()

    # Ground truth
    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
    all_item_indices = torch.arange(n_items).to(device)

    recalls, precisions, ndcgs, hits = [], [], [], []

    with torch.no_grad():
        for user_raw, true_items in test_user_items.items():
            if user_raw not in user2idx:
                continue

            u_idx = torch.tensor([user2idx[user_raw]], dtype=torch.long).to(device)
            true_indices = set(item2idx[i] for i in true_items if i in item2idx)

            if not true_indices:
                continue

            # Score all items
            if model_type == "mf":
                user_emb = model.user_emb(u_idx)
                scores   = (user_emb @ model.item_emb.weight.T).squeeze()
            elif model_type == "ncf":
                u_rep    = u_idx.repeat(n_items)
                scores   = model(u_rep, all_item_indices).squeeze()
            elif model_type == "lgcn":
                user_embs, item_embs = model.forward(adj)
                scores = (user_embs[u_idx] @ item_embs.T).squeeze()

            # Remove train items
            train_items = train_user_items.get(user_raw, set())
            train_indices = [item2idx[i] for i in train_items if i in item2idx]
            scores[train_indices] = -1e9

            # Top K
            topk = torch.topk(scores, K).indices.cpu().numpy()
            topk_set = set(topk)

            # Metrics
            hit        = len(topk_set & true_indices)
            recall     = hit / len(true_indices)
            precision  = hit / K
            hit_ratio  = 1.0 if hit > 0 else 0.0

            # NDCG
            dcg = 0
            for rank, idx in enumerate(topk):
                if idx in true_indices:
                    dcg += 1 / np.log2(rank + 2)
            idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_indices), K)))
            ndcg = dcg / idcg if idcg > 0 else 0

            recalls.append(recall)
            precisions.append(precision)
            ndcgs.append(ndcg)
            hits.append(hit_ratio)

    return {
        f"Recall@{K}"    : round(np.mean(recalls), 4),
        f"Precision@{K}" : round(np.mean(precisions), 4),
        f"NDCG@{K}"      : round(np.mean(ndcgs), 4),
        f"HitRatio@{K}"  : round(np.mean(hits), 4),
    }

print("Dataset and evaluation function ready.")

Using device: cpu
Dataset and evaluation function ready.


In [11]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Implicit Dataset ──────────────────────────────────────────────────────────
class ImplicitDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor([user2idx[u] for u in df["user_id"]], dtype=torch.long)
        self.items = torch.tensor([item2idx[i] for i in df["item_id"]], dtype=torch.long)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx]

train_loader = DataLoader(ImplicitDataset(train_df), batch_size=1024, shuffle=True)

# ── Negative sampling ─────────────────────────────────────────────────────────
idx2user      = {v: k for k, v in user2idx.items()}
idx2item      = {v: k for k, v in item2idx.items()}
train_pos     = train_df.groupby("user_id")["item_id"].apply(set).to_dict()
all_item_list = np.array(list(item2idx.keys()))
n_all_items   = len(all_item_list)

def sample_negatives_vectorized(user_indices):
    neg_indices = []
    for u_idx in user_indices.numpy():
        u_raw   = idx2user[u_idx]
        pos_set = train_pos.get(u_raw, set())
        while True:
            neg = all_item_list[np.random.randint(n_all_items)]
            if neg not in pos_set:
                neg_indices.append(item2idx[neg])
                break
    return torch.tensor(neg_indices, dtype=torch.long)

# ── BPR Loss ──────────────────────────────────────────────────────────────────
def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()

# ── Model ─────────────────────────────────────────────────────────────────────
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64):
        super().__init__()
        self.user_emb  = nn.Embedding(n_users, n_factors)
        self.item_emb  = nn.Embedding(n_items, n_factors)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, users, items):
        return (self.user_emb(users) * self.item_emb(items)).sum(dim=1)

# ── Train ─────────────────────────────────────────────────────────────────────
svd_model = MatrixFactorization(n_users, n_items, n_factors=64).to(device)
optimizer = torch.optim.Adam(svd_model.parameters(), lr=0.001, weight_decay=1e-5)

EPOCHS = 50
for epoch in range(EPOCHS):
    svd_model.train()
    total_loss = 0
    for users, items in train_loader:
        neg_items            = sample_negatives_vectorized(users.cpu()).to(device)
        users, items         = users.to(device), items.to(device)
        pos_scores           = svd_model(users, items)
        neg_scores           = svd_model(users, neg_items)
        loss                 = bpr_loss(pos_scores, neg_scores)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss          += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
train_user_items = train_df.groupby("user_id")["item_id"].apply(set).to_dict()

def evaluate(model, test_df, K=10, model_type="mf"):
    model.eval()
    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
    recalls, precisions, ndcgs, hits = [], [], [], []
    all_item_indices = torch.arange(n_items).to(device)

    with torch.no_grad():
        for user_raw, true_items in test_user_items.items():
            if user_raw not in user2idx:
                continue
            u_idx        = torch.tensor([user2idx[user_raw]], dtype=torch.long).to(device)
            true_indices = set(item2idx[i] for i in true_items if i in item2idx)
            if not true_indices:
                continue

            if model_type == "mf":
                scores = (model.user_emb(u_idx) @ model.item_emb.weight.T).squeeze()
            elif model_type == "ncf":
                u_rep  = u_idx.repeat(n_items)
                scores = model(u_rep, all_item_indices).squeeze()

            train_items       = train_user_items.get(user_raw, set())
            train_idx         = [item2idx[i] for i in train_items if i in item2idx]
            scores[train_idx] = -1e9

            topk     = torch.topk(scores, K).indices.cpu().numpy()
            topk_set = set(topk)
            hit      = len(topk_set & true_indices)

            recalls.append(hit / len(true_indices))
            precisions.append(hit / K)
            hits.append(1.0 if hit > 0 else 0.0)
            dcg  = sum(1 / np.log2(r + 2) for r, idx in enumerate(topk) if idx in true_indices)
            idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_indices), K)))
            ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return {
        f"Recall@{K}"    : round(np.mean(recalls), 4),
        f"Precision@{K}" : round(np.mean(precisions), 4),
        f"NDCG@{K}"      : round(np.mean(ndcgs), 4),
        f"HitRatio@{K}"  : round(np.mean(hits), 4),
    }

svd_results = evaluate(svd_model, test_df, K=10, model_type="mf")
svd_results["model"] = "SVD"
print(f"\nSVD Results: {svd_results}")

Using device: cpu
Epoch 1/50 | Loss: 0.5761
Epoch 2/50 | Loss: 0.3205
Epoch 3/50 | Loss: 0.2999
Epoch 4/50 | Loss: 0.2872
Epoch 5/50 | Loss: 0.2745
Epoch 6/50 | Loss: 0.2638
Epoch 7/50 | Loss: 0.2574
Epoch 8/50 | Loss: 0.2517
Epoch 9/50 | Loss: 0.2462
Epoch 10/50 | Loss: 0.2415
Epoch 11/50 | Loss: 0.2362
Epoch 12/50 | Loss: 0.2326
Epoch 13/50 | Loss: 0.2300
Epoch 14/50 | Loss: 0.2262
Epoch 15/50 | Loss: 0.2235
Epoch 16/50 | Loss: 0.2217
Epoch 17/50 | Loss: 0.2201
Epoch 18/50 | Loss: 0.2186
Epoch 19/50 | Loss: 0.2160
Epoch 20/50 | Loss: 0.2149
Epoch 21/50 | Loss: 0.2134
Epoch 22/50 | Loss: 0.2133
Epoch 23/50 | Loss: 0.2109
Epoch 24/50 | Loss: 0.2102
Epoch 25/50 | Loss: 0.2096
Epoch 26/50 | Loss: 0.2080
Epoch 27/50 | Loss: 0.2077
Epoch 28/50 | Loss: 0.2071
Epoch 29/50 | Loss: 0.2062
Epoch 30/50 | Loss: 0.2068
Epoch 31/50 | Loss: 0.2043
Epoch 32/50 | Loss: 0.2049
Epoch 33/50 | Loss: 0.2039
Epoch 34/50 | Loss: 0.2030
Epoch 35/50 | Loss: 0.2028
Epoch 36/50 | Loss: 0.2020
Epoch 37/50 | Loss:

In [12]:
import pickle

# Move model to CPU and re-save
svd_model_cpu = svd_model.cpu()

with open("svd_model_cpu.pkl", "wb") as f:
    pickle.dump(svd_model_cpu, f)

print("Saved CPU model")

Saved CPU model


In [17]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, layers=[128, 64, 32]):
        super().__init__()
        self.gmf_user = nn.Embedding(n_users, n_factors)
        self.gmf_item = nn.Embedding(n_items, n_factors)
        self.mlp_user = nn.Embedding(n_users, n_factors)
        self.mlp_item = nn.Embedding(n_items, n_factors)

        nn.init.xavier_uniform_(self.gmf_user.weight)
        nn.init.xavier_uniform_(self.gmf_item.weight)
        nn.init.xavier_uniform_(self.mlp_user.weight)
        nn.init.xavier_uniform_(self.mlp_item.weight)

        mlp_input_size = n_factors * 2
        mlp_layers = []
        for out_size in layers:
            mlp_layers.append(nn.Linear(mlp_input_size, out_size))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(0.2))
            mlp_input_size = out_size
        self.mlp = nn.Sequential(*mlp_layers)
        self.output = nn.Linear(n_factors + layers[-1], 1)

    def forward(self, users, items):
        gmf     = self.gmf_user(users) * self.gmf_item(items)
        mlp_in  = torch.cat([self.mlp_user(users), self.mlp_item(items)], dim=1)
        mlp_out = self.mlp(mlp_in)
        return self.output(torch.cat([gmf, mlp_out], dim=1)).squeeze()

# ── Train ─────────────────────────────────────────────────────────────────────
ncf_model = NCF(n_users, n_items, n_factors=64, layers=[128, 64, 32]).to(device)
optimizer = torch.optim.Adam(ncf_model.parameters(), lr=0.001, weight_decay=1e-5)

EPOCHS = 50
for epoch in range(EPOCHS):
    ncf_model.train()
    total_loss = 0
    for users, items in train_loader:
        neg_items = sample_negatives_vectorized(users.cpu()).to(device)
        users, items = users.to(device), items.to(device)

        pos_scores = ncf_model(users, items)
        neg_scores = ncf_model(users, neg_items)
        loss = bpr_loss(pos_scores, neg_scores)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
ncf_results = evaluate(ncf_model, test_df, K=10, model_type="ncf")
ncf_results["model"] = "NCF"
print(f"\nNCF Results: {ncf_results}")

Epoch 1/50 | Loss: 0.3660
Epoch 2/50 | Loss: 0.3145
Epoch 3/50 | Loss: 0.3117
Epoch 4/50 | Loss: 0.3107
Epoch 5/50 | Loss: 0.3073
Epoch 6/50 | Loss: 0.3041
Epoch 7/50 | Loss: 0.2953
Epoch 8/50 | Loss: 0.2730
Epoch 9/50 | Loss: 0.2571
Epoch 10/50 | Loss: 0.2487
Epoch 11/50 | Loss: 0.2392
Epoch 12/50 | Loss: 0.2303
Epoch 13/50 | Loss: 0.2221
Epoch 14/50 | Loss: 0.2172
Epoch 15/50 | Loss: 0.2118
Epoch 16/50 | Loss: 0.2090
Epoch 17/50 | Loss: 0.2045
Epoch 18/50 | Loss: 0.2022
Epoch 19/50 | Loss: 0.1987
Epoch 20/50 | Loss: 0.1965
Epoch 21/50 | Loss: 0.1932
Epoch 22/50 | Loss: 0.1913
Epoch 23/50 | Loss: 0.1878
Epoch 24/50 | Loss: 0.1867
Epoch 25/50 | Loss: 0.1840
Epoch 26/50 | Loss: 0.1827
Epoch 27/50 | Loss: 0.1802
Epoch 28/50 | Loss: 0.1785
Epoch 29/50 | Loss: 0.1766
Epoch 30/50 | Loss: 0.1747
Epoch 31/50 | Loss: 0.1722
Epoch 32/50 | Loss: 0.1712
Epoch 33/50 | Loss: 0.1700
Epoch 34/50 | Loss: 0.1701
Epoch 35/50 | Loss: 0.1683
Epoch 36/50 | Loss: 0.1659
Epoch 37/50 | Loss: 0.1651
Epoch 38/5

In [18]:
import scipy.sparse as sp

class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def build_adj_matrix(self, df, device):
        users = df["user_id"].map(user2idx).values
        items = df["item_id"].map(item2idx).values
        n     = self.n_users + self.n_items
        row   = np.concatenate([users, items + self.n_users])
        col   = np.concatenate([items + self.n_users, users])
        data  = np.ones(len(row))
        adj   = sp.coo_matrix((data, (row, col)), shape=(n, n))
        deg   = np.array(adj.sum(axis=1)).flatten()
        d_inv = np.power(deg, -0.5, where=deg > 0, out=np.zeros_like(deg, dtype=float))
        D     = sp.diags(d_inv)
        adj   = D @ adj @ D
        adj   = adj.tocoo()
        indices = torch.tensor(np.array([adj.row, adj.col]), dtype=torch.long)
        values  = torch.tensor(adj.data, dtype=torch.float32)
        return torch.sparse_coo_tensor(indices, values, (n, n)).to(device)

    def forward(self, adj):
        embs     = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        all_embs = [embs]
        for _ in range(self.n_layers):
            embs = torch.sparse.mm(adj, embs)
            all_embs.append(embs)
        all_embs = torch.stack(all_embs, dim=1).mean(dim=1)
        return all_embs[:self.n_users], all_embs[self.n_users:]

    def predict(self, users, items, adj):
        user_embs, item_embs = self.forward(adj)
        return (user_embs[users] * item_embs[items]).sum(dim=1)

# ── Build adj + Train ─────────────────────────────────────────────────────────
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
adj        = lgcn_model.build_adj_matrix(train_df, device)
optimizer  = torch.optim.Adam(lgcn_model.parameters(), lr=0.001, weight_decay=1e-5)

EPOCHS = 50
for epoch in range(EPOCHS):
    lgcn_model.train()
    total_loss = 0
    for users, items in train_loader:
        neg_items = sample_negatives_vectorized(users.cpu()).to(device)
        users, items = users.to(device), items.to(device)

        pos_scores = lgcn_model.predict(users, items, adj)
        neg_scores = lgcn_model.predict(users, neg_items, adj)
        loss       = bpr_loss(pos_scores, neg_scores)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

# ── Evaluate ──────────────────────────────────────────────────────────────────
lgcn_results = evaluate(lgcn_model, test_df, K=10, model_type="lgcn")
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")

Epoch 1/50 | Loss: 0.4274
Epoch 2/50 | Loss: 0.3452
Epoch 3/50 | Loss: 0.3442
Epoch 4/50 | Loss: 0.3436
Epoch 5/50 | Loss: 0.3445
Epoch 6/50 | Loss: 0.3433
Epoch 7/50 | Loss: 0.3427
Epoch 8/50 | Loss: 0.3429
Epoch 9/50 | Loss: 0.3429
Epoch 10/50 | Loss: 0.3434
Epoch 11/50 | Loss: 0.3422


KeyboardInterrupt: 

In [23]:
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
adj        = lgcn_model.build_adj_matrix(train_df, device)
optimizer  = torch.optim.Adam(lgcn_model.parameters(), lr=0.015, weight_decay=1e-4)

In [24]:
EPOCHS = 50
#optimizer = torch.optim.Adam(lgcn_model.parameters(), lr=0.005, weight_decay=1e-5)

for epoch in range(EPOCHS):
    lgcn_model.train()
    total_loss = 0

    for users, items in train_loader:
        neg_items = sample_negatives_vectorized(users.cpu()).to(device)
        users, items = users.to(device), items.to(device)

        # Forward through graph for this batch only
        all_user_emb = lgcn_model.user_emb.weight
        all_item_emb = lgcn_model.item_emb.weight
        embs = torch.cat([all_user_emb, all_item_emb], dim=0)

        # Light graph convolution
        all_embs = [embs]
        for _ in range(lgcn_model.n_layers):
            embs = torch.sparse.mm(adj, embs)
            all_embs.append(embs)
        all_embs = torch.stack(all_embs, dim=1).mean(dim=1)

        user_embs = all_embs[:n_users]
        item_embs = all_embs[n_users:]

        pos_scores = (user_embs[users] * item_embs[items]).sum(dim=1)
        neg_scores = (user_embs[users] * item_embs[neg_items]).sum(dim=1)
        loss       = bpr_loss(pos_scores, neg_scores)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

lgcn_results = evaluate(lgcn_model, test_df, K=10, model_type="lgcn")
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")

Epoch 1/50 | Loss: 0.5400
Epoch 2/50 | Loss: 0.5326
Epoch 3/50 | Loss: 0.5323
Epoch 4/50 | Loss: 0.5335
Epoch 5/50 | Loss: 0.5330
Epoch 6/50 | Loss: 0.5327
Epoch 7/50 | Loss: 0.5329
Epoch 8/50 | Loss: 0.5328
Epoch 9/50 | Loss: 0.5327
Epoch 10/50 | Loss: 0.5324
Epoch 11/50 | Loss: 0.5330
Epoch 12/50 | Loss: 0.5330
Epoch 13/50 | Loss: 0.5323
Epoch 14/50 | Loss: 0.5330


KeyboardInterrupt: 

In [25]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.2 MB/s eta 0:00:00


In [30]:
import scipy.sparse as sp
import torch
import torch.nn as nn
import numpy as np

# ── Build adjacency ───────────────────────────────────────────────────────────
def build_adj(train_df, n_users, n_items, device):
    users = np.array([user2idx[u] for u in train_df["user_id"]])
    items = np.array([item2idx[i] for i in train_df["item_id"]])
    R     = sp.csr_matrix((np.ones(len(users)), (users, items)), shape=(n_users, n_items))
    upper = sp.hstack([sp.csr_matrix((n_users, n_users)), R])
    lower = sp.hstack([R.T, sp.csr_matrix((n_items, n_items))])
    A     = sp.vstack([upper, lower]).tocsr()
    deg   = np.array(A.sum(1)).flatten()
    d_inv = np.zeros_like(deg)
    d_inv[deg > 0] = deg[deg > 0] ** -0.5
    D     = sp.diags(d_inv)
    A_hat = (D @ A @ D).tocoo()
    idx   = torch.tensor(np.array([A_hat.row, A_hat.col]), dtype=torch.long)
    val   = torch.tensor(A_hat.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(idx, val, A_hat.shape).coalesce().to(device)

# ── Model ─────────────────────────────────────────────────────────────────────
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_layers = n_layers
        self.E0       = nn.Embedding(n_users + n_items, n_factors)
        nn.init.normal_(self.E0.weight, std=0.1)

    def get_embeddings(self, adj):
        x   = self.E0.weight
        out = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(adj, x)
            out.append(x)
        out = torch.stack(out, dim=1).mean(dim=1)
        return out[:self.n_users], out[self.n_users:]

# ── BPR Loss ──────────────────────────────────────────────────────────────────
def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()

# ── Evaluate ──────────────────────────────────────────────────────────────────
def evaluate_lgcn(model, adj, test_df, K=10):
    model.eval()
    test_user_items = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
    recalls, precisions, ndcgs, hits = [], [], [], []

    with torch.no_grad():
        user_embs, item_embs = model.get_embeddings(adj)
        for user_raw, true_items in test_user_items.items():
            if user_raw not in user2idx:
                continue
            u_idx        = user2idx[user_raw]
            true_indices = set(item2idx[i] for i in true_items if i in item2idx)
            if not true_indices:
                continue
            scores              = (user_embs[u_idx] @ item_embs.T)
            train_items         = train_user_items.get(user_raw, set())
            train_idx           = [item2idx[i] for i in train_items if i in item2idx]
            scores[train_idx]   = -1e9
            topk                = torch.topk(scores, K).indices.cpu().numpy()
            topk_set            = set(topk)
            hit                 = len(topk_set & true_indices)
            recalls.append(hit / len(true_indices))
            precisions.append(hit / K)
            hits.append(1.0 if hit > 0 else 0.0)
            dcg  = sum(1 / np.log2(r + 2) for r, idx in enumerate(topk) if idx in true_indices)
            idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_indices), K)))
            ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return {
        f"Recall@{K}"    : round(np.mean(recalls), 4),
        f"Precision@{K}" : round(np.mean(precisions), 4),
        f"NDCG@{K}"      : round(np.mean(ndcgs), 4),
        f"HitRatio@{K}"  : round(np.mean(hits), 4),
    }

# ── Prepare full training tensors ─────────────────────────────────────────────
all_train_users = torch.tensor([user2idx[u] for u in train_df["user_id"]], dtype=torch.long)
all_train_items = torch.tensor([item2idx[i] for i in train_df["item_id"]], dtype=torch.long)

# ── Train ─────────────────────────────────────────────────────────────────────
adj        = build_adj(train_df, n_users, n_items, device)
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
optimizer  = torch.optim.Adam(lgcn_model.parameters(), lr=0.001, weight_decay=1e-4)

BATCH_SIZE = 4096
EPOCHS     = 50

for epoch in range(EPOCHS):
    lgcn_model.train()
    total_loss = 0
    n_batches  = 0

    # Compute full graph embeddings once per epoch
    user_embs, item_embs = lgcn_model.get_embeddings(adj)

    # Shuffle training data
    perm = torch.randperm(len(all_train_users))
    shuffled_users = all_train_users[perm]
    shuffled_items = all_train_items[perm]

    for start in range(0, len(shuffled_users), BATCH_SIZE):
        batch_users = shuffled_users[start:start+BATCH_SIZE].to(device)
        batch_items = shuffled_items[start:start+BATCH_SIZE].to(device)
        neg_items   = sample_negatives_vectorized(batch_users.cpu()).to(device)

        pos_scores = (user_embs[batch_users] * item_embs[batch_items]).sum(dim=1)
        neg_scores = (user_embs[batch_users] * item_embs[neg_items]).sum(dim=1)
        loss       = bpr_loss(pos_scores, neg_scores)

        optimizer.zero_grad()
        loss.backward(retain_graph=True)
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/n_batches:.4f}")

lgcn_results = evaluate_lgcn(lgcn_model, adj, test_df, K=10)
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")

Epoch 1/50 | Loss: 0.6928
Epoch 2/50 | Loss: 0.6930
Epoch 3/50 | Loss: 0.6929
Epoch 4/50 | Loss: 0.6924
Epoch 5/50 | Loss: 0.6912
Epoch 6/50 | Loss: 0.6878
Epoch 7/50 | Loss: 0.6789
Epoch 8/50 | Loss: 0.6576
Epoch 9/50 | Loss: 0.6151
Epoch 10/50 | Loss: 0.5613
Epoch 11/50 | Loss: 0.5340
Epoch 12/50 | Loss: 0.5324
Epoch 13/50 | Loss: 0.5314
Epoch 14/50 | Loss: 0.5316
Epoch 15/50 | Loss: 0.5318
Epoch 16/50 | Loss: 0.5323
Epoch 17/50 | Loss: 0.5310
Epoch 18/50 | Loss: 0.5319
Epoch 19/50 | Loss: 0.5321
Epoch 20/50 | Loss: 0.5324
Epoch 21/50 | Loss: 0.5322
Epoch 22/50 | Loss: 0.5312
Epoch 23/50 | Loss: 0.5325
Epoch 24/50 | Loss: 0.5316
Epoch 25/50 | Loss: 0.5323
Epoch 26/50 | Loss: 0.5322
Epoch 27/50 | Loss: 0.5320
Epoch 28/50 | Loss: 0.5324


KeyboardInterrupt: 

In [31]:
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
optimizer  = torch.optim.Adam(lgcn_model.parameters(), lr=0.001, weight_decay=1e-4)

In [34]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, n_factors=64, n_layers=3):
        super().__init__()
        self.n_users  = n_users
        self.n_layers = n_layers
        self.E0       = nn.Embedding(n_users + n_items, n_factors)
        nn.init.normal_(self.E0.weight, std=0.1)

    def get_embeddings(self, adj_dense):
        x   = self.E0.weight
        out = [x]
        for _ in range(self.n_layers):
            x = adj_dense @ x      # dense mm — gradients flow properly
            out.append(x)
        out = torch.stack(out, dim=1).mean(dim=1)
        return out[:self.n_users], out[self.n_users:]

# Build DENSE adj matrix — fits in GPU memory for this dataset size
def build_adj_dense(train_df, n_users, n_items, device):
    users = np.array([user2idx[u] for u in train_df["user_id"]])
    items = np.array([item2idx[i] for i in train_df["item_id"]])
    R     = sp.csr_matrix((np.ones(len(users)), (users, items)), shape=(n_users, n_items))
    upper = sp.hstack([sp.csr_matrix((n_users, n_users)), R])
    lower = sp.hstack([R.T, sp.csr_matrix((n_items, n_items))])
    A     = sp.vstack([upper, lower]).tocsr()
    deg   = np.array(A.sum(1)).flatten()
    d_inv = np.zeros_like(deg)
    d_inv[deg > 0] = deg[deg > 0] ** -0.5
    D     = sp.diags(d_inv)
    A_hat = (D @ A @ D)
    return torch.tensor(A_hat.toarray(), dtype=torch.float32).to(device)

adj_dense  = build_adj_dense(train_df, n_users, n_items, device)
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
optimizer  = torch.optim.Adam(lgcn_model.parameters(), lr=0.001, weight_decay=1e-4)

BATCH_SIZE = 4096
EPOCHS     = 50

for epoch in range(EPOCHS):
    lgcn_model.train()
    optimizer.zero_grad()

    user_embs, item_embs = lgcn_model.get_embeddings(adj_dense)

    perm        = torch.randperm(len(all_train_users))[:BATCH_SIZE]
    batch_users = all_train_users[perm].to(device)
    batch_items = all_train_items[perm].to(device)
    neg_items   = sample_negatives_vectorized(batch_users.cpu()).to(device)

    pos_scores = (user_embs[batch_users] * item_embs[batch_items]).sum(dim=1)
    neg_scores = (user_embs[batch_users] * item_embs[neg_items]).sum(dim=1)
    loss       = bpr_loss(pos_scores, neg_scores)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f}")

lgcn_results = evaluate_lgcn(lgcn_model, adj_dense, test_df, K=10)
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")

Epoch 1/50 | Loss: 0.6928
Epoch 2/50 | Loss: 0.6927
Epoch 3/50 | Loss: 0.6929
Epoch 4/50 | Loss: 0.6929
Epoch 5/50 | Loss: 0.6929
Epoch 6/50 | Loss: 0.6928
Epoch 7/50 | Loss: 0.6930
Epoch 8/50 | Loss: 0.6929
Epoch 9/50 | Loss: 0.6929
Epoch 10/50 | Loss: 0.6928
Epoch 11/50 | Loss: 0.6929
Epoch 12/50 | Loss: 0.6929
Epoch 13/50 | Loss: 0.6929
Epoch 14/50 | Loss: 0.6929
Epoch 15/50 | Loss: 0.6929
Epoch 16/50 | Loss: 0.6929
Epoch 17/50 | Loss: 0.6929
Epoch 18/50 | Loss: 0.6929
Epoch 19/50 | Loss: 0.6929
Epoch 20/50 | Loss: 0.6929
Epoch 21/50 | Loss: 0.6929
Epoch 22/50 | Loss: 0.6929
Epoch 23/50 | Loss: 0.6929
Epoch 24/50 | Loss: 0.6929
Epoch 25/50 | Loss: 0.6929
Epoch 26/50 | Loss: 0.6930
Epoch 27/50 | Loss: 0.6929
Epoch 28/50 | Loss: 0.6929
Epoch 29/50 | Loss: 0.6929
Epoch 30/50 | Loss: 0.6930
Epoch 31/50 | Loss: 0.6929
Epoch 32/50 | Loss: 0.6929
Epoch 33/50 | Loss: 0.6929
Epoch 34/50 | Loss: 0.6929
Epoch 35/50 | Loss: 0.6929
Epoch 36/50 | Loss: 0.6929
Epoch 37/50 | Loss: 0.6929
Epoch 38/5

In [35]:
print("adj_dense shape:", adj_dense.shape)
print("adj_dense sum:", adj_dense.sum().item())
print("adj_dense max:", adj_dense.max().item())
print("E0 grad after backward:", lgcn_model.E0.weight.grad is not None)
print("Sample scores - pos:", pos_scores[:5].detach().cpu().numpy())
print("Sample scores - neg:", neg_scores[:5].detach().cpu().numpy())

adj_dense shape: torch.Size([9571, 9571])
adj_dense sum: 5980.06884765625
adj_dense max: 0.37796446681022644
E0 grad after backward: True
Sample scores - pos: [0.00371891 0.00346046 0.00284803 0.00485781 0.00016096]
Sample scores - neg: [ 7.3448988e-03  8.0629991e-04 -9.9346711e-05 -3.7495107e-03
 -2.3129187e-03]


In [36]:
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)

# Reinitialize with larger std
nn.init.normal_(lgcn_model.E0.weight, std=1.0)

optimizer = torch.optim.Adam(lgcn_model.parameters(), lr=0.01, weight_decay=1e-4)

BATCH_SIZE = 4096
EPOCHS     = 50

for epoch in range(EPOCHS):
    lgcn_model.train()
    optimizer.zero_grad()

    user_embs, item_embs = lgcn_model.get_embeddings(adj_dense)

    perm        = torch.randperm(len(all_train_users))[:BATCH_SIZE]
    batch_users = all_train_users[perm].to(device)
    batch_items = all_train_items[perm].to(device)
    neg_items   = sample_negatives_vectorized(batch_users.cpu()).to(device)

    pos_scores = (user_embs[batch_users] * item_embs[batch_items]).sum(dim=1)
    neg_scores = (user_embs[batch_users] * item_embs[neg_items]).sum(dim=1)
    loss       = bpr_loss(pos_scores, neg_scores)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f}")

lgcn_results = evaluate_lgcn(lgcn_model, adj_dense, test_df, K=10)
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")

Epoch 1/50 | Loss: 0.7280
Epoch 2/50 | Loss: 0.7195
Epoch 3/50 | Loss: 0.7202
Epoch 4/50 | Loss: 0.7180
Epoch 5/50 | Loss: 0.7255
Epoch 6/50 | Loss: 0.7169
Epoch 7/50 | Loss: 0.7168
Epoch 8/50 | Loss: 0.7134
Epoch 9/50 | Loss: 0.7108
Epoch 10/50 | Loss: 0.7089
Epoch 11/50 | Loss: 0.7076
Epoch 12/50 | Loss: 0.7066
Epoch 13/50 | Loss: 0.7035
Epoch 14/50 | Loss: 0.7072
Epoch 15/50 | Loss: 0.7128
Epoch 16/50 | Loss: 0.7089
Epoch 17/50 | Loss: 0.7120
Epoch 18/50 | Loss: 0.7077
Epoch 19/50 | Loss: 0.7001
Epoch 20/50 | Loss: 0.7074
Epoch 21/50 | Loss: 0.7037
Epoch 22/50 | Loss: 0.6993
Epoch 23/50 | Loss: 0.6973
Epoch 24/50 | Loss: 0.7040
Epoch 25/50 | Loss: 0.6971
Epoch 26/50 | Loss: 0.6994
Epoch 27/50 | Loss: 0.7012
Epoch 28/50 | Loss: 0.6943
Epoch 29/50 | Loss: 0.6883
Epoch 30/50 | Loss: 0.6928
Epoch 31/50 | Loss: 0.6970
Epoch 32/50 | Loss: 0.7040
Epoch 33/50 | Loss: 0.6977
Epoch 34/50 | Loss: 0.6952
Epoch 35/50 | Loss: 0.6916
Epoch 36/50 | Loss: 0.6899
Epoch 37/50 | Loss: 0.6911
Epoch 38/5

In [38]:
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
nn.init.normal_(lgcn_model.E0.weight, std=0.1)
optimizer  = torch.optim.Adam(lgcn_model.parameters(), lr=0.001, weight_decay=1e-4)

BATCH_SIZE = 4096
EPOCHS     = 50

for epoch in range(EPOCHS):
    lgcn_model.train()

    # Single forward pass for the whole epoch
    user_embs, item_embs = lgcn_model.get_embeddings(adj_dense)

    epoch_loss = 0
    optimizer.zero_grad()

    perm = torch.randperm(len(all_train_users))
    for start in range(0, len(all_train_users), BATCH_SIZE):
        batch_users = all_train_users[perm[start:start+BATCH_SIZE]].to(device)
        batch_items = all_train_items[perm[start:start+BATCH_SIZE]].to(device)
        neg_items   = sample_negatives_vectorized(batch_users.cpu()).to(device)

        pos_scores = (user_embs[batch_users] * item_embs[batch_items]).sum(dim=1)
        neg_scores = (user_embs[batch_users] * item_embs[neg_items]).sum(dim=1)
        loss       = bpr_loss(pos_scores, neg_scores)

        # Accumulate gradients, don't step yet
        loss.backward(retain_graph=True)
        epoch_loss += loss.item()

    # Single optimizer step per epoch
    optimizer.step()
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss / (len(all_train_users) // BATCH_SIZE):.4f}")

lgcn_results = evaluate_lgcn(lgcn_model, adj_dense, test_df, K=10)
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")

Epoch 1/50 | Loss: 0.6990
Epoch 2/50 | Loss: 0.6990
Epoch 3/50 | Loss: 0.6989
Epoch 4/50 | Loss: 0.6987
Epoch 5/50 | Loss: 0.6986
Epoch 6/50 | Loss: 0.6984
Epoch 7/50 | Loss: 0.6981
Epoch 8/50 | Loss: 0.6978
Epoch 9/50 | Loss: 0.6975
Epoch 10/50 | Loss: 0.6971
Epoch 11/50 | Loss: 0.6966
Epoch 12/50 | Loss: 0.6961
Epoch 13/50 | Loss: 0.6955
Epoch 14/50 | Loss: 0.6949
Epoch 15/50 | Loss: 0.6942
Epoch 16/50 | Loss: 0.6935
Epoch 17/50 | Loss: 0.6926
Epoch 18/50 | Loss: 0.6917
Epoch 19/50 | Loss: 0.6907
Epoch 20/50 | Loss: 0.6896
Epoch 21/50 | Loss: 0.6885
Epoch 22/50 | Loss: 0.6872
Epoch 23/50 | Loss: 0.6859
Epoch 24/50 | Loss: 0.6845
Epoch 25/50 | Loss: 0.6830
Epoch 26/50 | Loss: 0.6815
Epoch 27/50 | Loss: 0.6798
Epoch 28/50 | Loss: 0.6780
Epoch 29/50 | Loss: 0.6761
Epoch 30/50 | Loss: 0.6742
Epoch 31/50 | Loss: 0.6722
Epoch 32/50 | Loss: 0.6700
Epoch 33/50 | Loss: 0.6678
Epoch 34/50 | Loss: 0.6655
Epoch 35/50 | Loss: 0.6631
Epoch 36/50 | Loss: 0.6606
Epoch 37/50 | Loss: 0.6580
Epoch 38/5

In [39]:
lgcn_model = LightGCN(n_users, n_items, n_factors=64, n_layers=3).to(device)
nn.init.normal_(lgcn_model.E0.weight, std=0.1)
optimizer  = torch.optim.Adam(lgcn_model.parameters(), lr=0.001, weight_decay=1e-4)

BATCH_SIZE = 4096
EPOCHS     = 200

for epoch in range(EPOCHS):
    lgcn_model.train()
    user_embs, item_embs = lgcn_model.get_embeddings(adj_dense)
    epoch_loss = 0
    optimizer.zero_grad()

    perm = torch.randperm(len(all_train_users))
    for start in range(0, len(all_train_users), BATCH_SIZE):
        batch_users = all_train_users[perm[start:start+BATCH_SIZE]].to(device)
        batch_items = all_train_items[perm[start:start+BATCH_SIZE]].to(device)
        neg_items   = sample_negatives_vectorized(batch_users.cpu()).to(device)

        pos_scores = (user_embs[batch_users] * item_embs[batch_items]).sum(dim=1)
        neg_scores = (user_embs[batch_users] * item_embs[neg_items]).sum(dim=1)
        loss       = bpr_loss(pos_scores, neg_scores)
        loss.backward(retain_graph=True)
        epoch_loss += loss.item()

    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss / (len(all_train_users) // BATCH_SIZE):.4f}")

lgcn_results = evaluate_lgcn(lgcn_model, adj_dense, test_df, K=10)
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")

Epoch 10/200 | Loss: 0.6971
Epoch 20/200 | Loss: 0.6894
Epoch 30/200 | Loss: 0.6738
Epoch 40/200 | Loss: 0.6491
Epoch 50/200 | Loss: 0.6157
Epoch 60/200 | Loss: 0.5766
Epoch 70/200 | Loss: 0.5349
Epoch 80/200 | Loss: 0.4943
Epoch 90/200 | Loss: 0.4582
Epoch 100/200 | Loss: 0.4279
Epoch 110/200 | Loss: 0.4032
Epoch 120/200 | Loss: 0.3852
Epoch 130/200 | Loss: 0.3710
Epoch 140/200 | Loss: 0.3606
Epoch 150/200 | Loss: 0.3526
Epoch 160/200 | Loss: 0.3478
Epoch 170/200 | Loss: 0.3424
Epoch 180/200 | Loss: 0.3389
Epoch 190/200 | Loss: 0.3367
Epoch 200/200 | Loss: 0.3339

LightGCN Results: {'Recall@10': np.float64(0.0822), 'Precision@10': np.float64(0.0675), 'NDCG@10': np.float64(0.0925), 'HitRatio@10': np.float64(0.4324), 'model': 'LightGCN'}


In [40]:
EPOCHS = 200

for epoch in range(EPOCHS):
    lgcn_model.train()
    user_embs, item_embs = lgcn_model.get_embeddings(adj_dense)
    epoch_loss = 0
    optimizer.zero_grad()

    perm = torch.randperm(len(all_train_users))
    for start in range(0, len(all_train_users), BATCH_SIZE):
        batch_users = all_train_users[perm[start:start+BATCH_SIZE]].to(device)
        batch_items = all_train_items[perm[start:start+BATCH_SIZE]].to(device)
        neg_items   = sample_negatives_vectorized(batch_users.cpu()).to(device)

        pos_scores = (user_embs[batch_users] * item_embs[batch_items]).sum(dim=1)
        neg_scores = (user_embs[batch_users] * item_embs[neg_items]).sum(dim=1)
        loss       = bpr_loss(pos_scores, neg_scores)
        loss.backward(retain_graph=True)
        epoch_loss += loss.item()

    optimizer.step()
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss / (len(all_train_users) // BATCH_SIZE):.4f}")

lgcn_results = evaluate_lgcn(lgcn_model, adj_dense, test_df, K=10)
lgcn_results["model"] = "LightGCN"
print(f"\nLightGCN Results: {lgcn_results}")

Epoch 20/200 | Loss: 0.3315
Epoch 40/200 | Loss: 0.3300
Epoch 60/200 | Loss: 0.3277
Epoch 80/200 | Loss: 0.3260
Epoch 100/200 | Loss: 0.3248
Epoch 120/200 | Loss: 0.3232
Epoch 140/200 | Loss: 0.3242
Epoch 160/200 | Loss: 0.3224
Epoch 180/200 | Loss: 0.3226
Epoch 200/200 | Loss: 0.3224

LightGCN Results: {'Recall@10': np.float64(0.0825), 'Precision@10': np.float64(0.0678), 'NDCG@10': np.float64(0.0928), 'HitRatio@10': np.float64(0.4322), 'model': 'LightGCN'}


In [41]:
import pickle, json

# Save SVD model
with open("svd_model.pkl", "wb") as f:
    pickle.dump(svd_model, f)

# Save encoders
with open("encoders.pkl", "wb") as f:
    pickle.dump({"user2idx": user2idx, "item2idx": item2idx,
                 "idx2user": idx2user, "idx2item": idx2item}, f)

# Save results
results = [svd_results, ncf_results, lgcn_results]
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved: svd_model.pkl, encoders.pkl, results.json")

Saved: svd_model.pkl, encoders.pkl, results.json


In [42]:
import pandas as pd

movies = pd.read_csv("ml-1m/movies.dat", sep="::", header=None,
                     names=["item_id", "title", "genres"], engine="python",
                     encoding="latin-1")
movies.to_csv("movies.csv", index=False)
print(movies.head())

   item_id                               title                        genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
